A loss turns predictions and observations into an objective for parameter
search. Fitters consume the `LossFn` interface, so the same fitting procedure
can use sequence likelihood or a discrepancy between behavioral summaries.

## The loss interface

Declaration from [`jaxcmr.typing`](https://github.com/githubpsyche/jaxcmr/blob/main/jaxcmr/typing.py):

```python
class LossFn(Protocol):
    def __call__(
        self,
        trial_indices: Integer[Array, " trials"],
        base_params: Parameters,
        free_param_names: Iterable[str],
        x: Float[Array, " parameters n_samples"],
    ) -> Float[Array, " n_samples"]: ...
```

The constructor binds the dataset, optional features and model factory.
A compatible class or configured callable is supplied as `loss_fn_cls`.
The `LossFactory` protocol describes construction; `LossFn` describes the
resulting population scorer. For example, `functools.partial` can bind MSE
analyses, repetition count and seed before fitting.

| Call argument | Responsibility |
|---|---|
| `trial_indices` | Explicit dataset row indices to score, rather than a Boolean mask. |
| `base_params` | Fixed values and defaults. Free values override matching entries. |
| `free_param_names` | Names in the order of the candidate matrix's parameter axis. |
| `x` | Physical model parameters shaped `(parameters, candidates)`. Each column is one candidate. |

The result has one objective per candidate. Fitters minimize these values.
A custom loss must preserve this argument order, parameter mapping and
output shape and support the JAX operations used by its chosen fitter.
Search-coordinate transformations belong to the fitting adapter; the loss
receives model parameter values.

## Likelihood and event masks

Sequence likelihood follows the observed response history, obtains the
probability of each event and sums its negative log probability. A
`LikelihoodMaskFn` chooses which events contribute to that sum.

Declaration from [`jaxcmr.typing`](https://github.com/githubpsyche/jaxcmr/blob/main/jaxcmr/typing.py):

```python
class LikelihoodMaskFn(Protocol):
    def __call__(
        self, recalls: Integer[Array, " recall_events"]
    ) -> Bool[Array, " recall_events"]: ...
```

In `jaxcmr.loss.sequence_likelihood`, `MemorySearchLikelihoodLoss`
accepts this callback as `mask_likelihoods`. It translates recorded study
positions to within-list item codes before calling the mask. The existing
`mask_trailing_terminations()` returns `recalls != 0`.

The convenient `ExcludeTerminationLikelihoodLoss` binds that mask and
provides the standard three-argument `LossFn` constructor used by fitters.
With the no-stop model used here, the objective scores the recorded item
sequence without scoring its length. Zero-coded slots do not contribute.
An excluded event does not erase the
preceding history that conditions later predictions. A first-recall mask,
for example, removes that event's score while later events still follow it.

In [ ]:
#| code-summary: "0 — Locate the project inputs"

from pathlib import Path

from jaxcmr.helpers import find_project_root, resolve_project_path

try:
    root_path = find_project_root()
except FileNotFoundError:
    root_path = Path.cwd()


In [1]:
#| code-summary: "Evaluate two parameter columns through the loss interface"

from pathlib import Path
import json
import jax
import jax.numpy as jnp
from jaxcmr.helpers import load_data
from jaxcmr.dataset import generate_trial_mask
from jaxcmr.components import context, linear_memory
from jaxcmr.components.termination import NoStopTermination
from jaxcmr.models.cmr import make_factory

jax.config.update("jax_enable_x64", True)
data = load_data(resolve_project_path("data/Dupertuys2026.h5", root_path))
record = json.loads(Path(resolve_project_path("data/fits/Dupertuys2026_CMR_demo.json", root_path)).read_text())
parameters = {name: values[0] for name, values in record["fits"].items()
              if name != "subject"}
for name in ("allow_repeated_recalls", "learn_after_context_update"):
    parameters[name] = bool(parameters[name])

CMRFactory = make_factory(
    mfc_create_fn=linear_memory.init_mfc,
    mcf_create_fn=linear_memory.init_mcf,
    context_create_fn=context.init,
    termination_policy_create_fn=NoStopTermination,
)

from jaxcmr.loss.sequence_likelihood import ExcludeTerminationLikelihoodLoss

mixed = generate_trial_mask(data, "data['list_type'] == 1")
trial_indices = jnp.flatnonzero(mixed)[:1]
free_names = ("encoding_drift_rate", "recall_drift_rate")
base_params = {name: value for name, value in parameters.items() if name not in free_names}
candidates = jnp.array([
    [parameters["encoding_drift_rate"], 0.2],
    [parameters["recall_drift_rate"], parameters["recall_drift_rate"]],
])

loss = ExcludeTerminationLikelihoodLoss(CMRFactory, data, features=None)
values = loss(trial_indices, base_params, free_names, candidates)
print("Parameter order:", free_names)
print("Candidate matrix shape:", candidates.shape)
print("Objective shape:", values.shape)
print("Negative log likelihoods:", values)

Parameter order: ('encoding_drift_rate', 'recall_drift_rate')
Candidate matrix shape: (2, 2)
Objective shape: (2,)
Negative log likelihoods: [13.62458774 13.05021919]


Only one mixed list is scored here to expose the interface. The first column
uses the supplied fit's parameter values; the second changes encoding drift.
The returned values are not the objectives from fitting the whole experiment.

The underlying likelihood also exposes event probabilities. Setting masked
probabilities to one makes their contribution `-log(1) = 0`.

In [2]:
#| code-summary: "Connect the event probabilities and mask to the returned NLL"

from jaxcmr.math import negative_log_likelihood
from jaxcmr.loss.sequence_likelihood import (
    MemorySearchLikelihoodLoss, mask_trailing_terminations,
)

likelihood = MemorySearchLikelihoodLoss(
    CMRFactory, data, features=None,
    mask_likelihoods=mask_trailing_terminations,
)
row = trial_indices[0]
probabilities = likelihood.predict_trial(row, parameters)
include = mask_trailing_terminations(data["recalls"][row])
event_nll = negative_log_likelihood(jnp.where(include, probabilities, 1.0))
print("Included event probabilities:", probabilities[include])
print("Included events:", int(include.sum()))
print("NLL from event probabilities:", float(event_nll))
print("NLL from the loss call:", float(values[0]))

Included event probabilities: [0.13703638 0.04187474 0.08562368 0.19103804 0.0785633  0.16413277]
Included events: 6
NLL from event probabilities: 13.624587741452169
NLL from the loss call: 13.62458774145214


`jaxcmr.math.negative_log_likelihood()` sums the negative log probabilities
of the included events. The mask selects the positive recall events; it does
not define a new response sequence.

## Behavioral-summary objectives

`jaxcmr.loss.mse.make_factory()` binds named `RecallAnalysisFn` callbacks into
a loss class with the same fitting interface. It compares observed and
simulated summaries rather than assigning a likelihood to each recorded
response. [Recall Analyses](analyses.ipynb#recallanalysisfn) shows the callback
contract and uses the existing SPC calculation directly.

Its weights, scales and reduction determine how summary discrepancies
contribute to the objective. Such a value is not NLL and cannot be put into
an AIC formula. JAX-compatible simulation also does not by itself provide
useful pathwise gradients through discrete recall samples.

## Selecting observations

Ordinary losses receive the selected trial indices on every call. Row order
and repeated indices are retained; selecting another participant does not
change a previously supplied selection. Fitters call this same loss interface.


The ordinary interface is:

```python
loss(trial_indices, base_params, free_param_names, candidates)
```

It returns one objective value per candidate column.


There is no preparation call before ordinary fitting. Gradient and
evolutionary fitters pass trial indices to stable compiled functions;
compatible shapes can reuse compilation while scoring different observations.


In [ ]:
#| code-summary: "Evaluate the same candidates on the selected trials"

loss(trial_indices, base_params, free_names, candidates)

[Fitting Procedures](fitting.ipynb) explains how fitters consume these losses;
the [fitting template](../templates/fitting.ipynb) carries out a full run.

OptimizedCMR has a separate, experimental prepared-sequence implementation.
Its acceleration is selected through explicit experimental fitting imports,
described under [Experimental OptimizedCMR](models.ipynb#experimental-optimizedcmr).
It adds no preparation requirements to ordinary losses or model factories.


## Behavioral summaries and unordered recall

`MemorySearchMseLoss` calls the same dataset analyses on observations and
simulations. One simulated dataset per candidate is shared across analyses.
`MemorySearchSpcMseLoss` uses `spc`; `MemorySearchCatSpcMseLoss` stacks `cat_spc`
results for explicit category values (defaults: field `condition`, values 1 and 2).
Undefined observed bins are omitted. A missing prediction for an observed bin
makes the fit invalid. Duplicate responses and repeated presentations follow
those analyses' definitions. Metadata and study-position recall codes stay aligned.

Simulation defaults are 20 repetitions for general/SPC MSE and 10 for category
SPC, with seed zero. Both are configurable. Stopping remains active; responses
after the observed response count are masked. This objective uses discrete
simulation, so it is not a suitable pathwise-gradient loss.

Experimental `RecallSetLikelihoodLoss` sums over unordered item sequences and
one final STOP. `RecallPrefixLikelihoodLoss` omits STOP. Both estimate the sum
by sampling permutations and multiplying by the number of distinct orderings,
including repeated-item multiplicities. The logarithm is also an approximation.
Old permutation and MSE scores do not represent these corrected definitions.

See [sequential scoring](../reference/loss-sequence.qmd),
[behavioral MSE](../reference/loss-mse.qmd) and
[permutation likelihood](../reference/experimental-permutation.qmd).
